# LightRet â€” Stage 1: BERT â†’ RetBERT Sentence Distillation

**Goal**: Teach RetBERT (word-level, RetVec backbone) to produce sentence embeddings
that match frozen BERT-Base. Both models process clean text.  
**Loss**: `1 âˆ’ cosine_similarity(z_RetBERT, z_BERT)` averaged over the batch.

## Required Kaggle Datasets (add before running)
| Dataset name | Contents |
|---|---|
| `lightret-source` | Upload the entire `lightret/` project folder (src/, train_*.py, etc.) |
| `lightret-weights` | Upload `retvec_v1_weights.npz` |

> **Internet**: must be **ON** â€” HuggingFace downloads BERT weights, WikiText-103, CoNLL-2003.

## Expected runtime (T4 GPU)
- Dataset loading: ~5 min (WikiText-103 is ~500 MB)
- Per epoch: ~90â€“120 min (WikiText sentences dominate)
- Total 5 epochs: ~8â€“10 hours  
  â†’ Consider reducing `STAGE1_EPOCHS` to 2â€“3 for a first run.

## Output
`/kaggle/working/weights/retbert_stage1.pt` â€” download and use as input for Stage 2.

In [ ]:
# â”€â”€ 1. Install packages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# torch, numpy are pre-installed on Kaggle
!pip install -q transformers datasets

In [ ]:
# â”€â”€ 2. Setup working directory â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Copies the project source and weights into /kaggle/working/ so that
# `import src.*` works and config.py ROOT resolves correctly.

import os, shutil, pathlib

# !! Adjust these if your Kaggle dataset names differ !!
SOURCE_DATASET  = '/kaggle/input/lightret-source'   # contains src/, train_*.py
WEIGHTS_DATASET = '/kaggle/input/lightret-weights'  # contains retvec_v1_weights.npz

WORK = pathlib.Path('/kaggle/working')

# Copy src/ to working dir
src_dst = WORK / 'src'
if src_dst.exists():
    shutil.rmtree(src_dst)
shutil.copytree(f'{SOURCE_DATASET}/src', str(src_dst))

# Copy RetVec weights
shutil.copy(f'{WEIGHTS_DATASET}/retvec_v1_weights.npz', str(WORK / 'retvec_v1_weights.npz'))

# Create weights output directory
(WORK / 'weights').mkdir(exist_ok=True)

os.chdir(str(WORK))
print('Working directory:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))


# -- Write latest dataset.py from embedded source (no GitHub needed) --
import base64 as _b64
_D = (
    "..."
)
with open("/kaggle/working/src/data/dataset.py", "wb") as _f:
    _f.write(_b64.decodebytes(_D.encode()))
print("dataset.py: written from embedded source")

# -- Download conll2003_local.zip from GitHub if not already available --------
import zipfile, pathlib, urllib.request as _ur

_conll_dst = pathlib.Path('/kaggle/working/conll2003_local')
_conll_candidates = [
    pathlib.Path('/kaggle/input/conll2003-local/conll2003_local'),
    pathlib.Path('/kaggle/input/datasets/maheshbadari/lightret-source/lightret/conll2003_local/conll2003_local'),
    _conll_dst,
]
_conll_found = any(p.exists() for p in _conll_candidates)

if not _conll_found:
    _zip_path = pathlib.Path('/kaggle/working/conll2003_local.zip')
    if not _zip_path.exists():
        print('conll2003_local.zip: downloading from GitHub ...')
        try:
            _ur.urlretrieve(
                'https://raw.githubusercontent.com/maheshbadari/LightRET/main/conll2003_local.zip',
                str(_zip_path)
            )
            print(f'  downloaded ({_zip_path.stat().st_size/1024:.0f} KB)')
        except Exception as e:
            print(f'  download failed: {e}')
    if _zip_path.exists():
        print('conll2003_local.zip: extracting ...')
        with zipfile.ZipFile(_zip_path) as zf:
            zf.extractall('/kaggle/working')
        print(f'  extracted to {_conll_dst}')
    else:
        print('WARNING: conll2003_local not available — will fall back to HuggingFace parquet')
else:
    print('conll2003_local: already available on disk')

In [ ]:
# ── Patch stale config.py before importing ───────────────────────────────────
# Fixes uploaded dataset having old dataset name (conll2003 -> eriktks/conll2003)
p = '/kaggle/working/src/config.py'
with open(p) as f:
    txt = f.read()
txt = txt.replace('"conll2003"', '"eriktks/conll2003"')
with open(p, 'w') as f:
    f.write(txt)
print('config.py patched — conll dataset:', end=" ")
[print(l.strip()) for l in txt.splitlines() if "conll2003" in l and "DATASET" in l]

In [ ]:
# â”€â”€ 3. Verify GPU â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# â”€â”€ 4. Verify imports â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
sys.path.insert(0, '/kaggle/working')

from src.config import DEVICE, RETVEC_WEIGHTS, WEIGHTS_DIR, STAGE1_CKPT
from src.models.retbert import RetBERT
from src.data.dataset import PretrainDataset, pretrain_collate
from src.losses import stage1_loss

print('Device          :', DEVICE)
print('RetVec weights  :', RETVEC_WEIGHTS, '| exists:', RETVEC_WEIGHTS.exists())
print('Checkpoint path :', STAGE1_CKPT)

# Quick forward-pass check with a tiny batch
model = RetBERT().to(DEVICE)
with torch.no_grad():
    _, pooled = model([['hello', 'world'], ['test']])
print('RetBERT forward OK â€” pooled shape:', tuple(pooled.shape))
del model

In [ ]:
# â”€â”€ 5. Hyperparameters (override config defaults here if needed) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import src.config as cfg

# Uncomment to reduce epochs for a quick test run
# cfg.STAGE1_EPOCHS     = 2
# cfg.STAGE1_BATCH_SIZE = 16   # lower if OOM

print(f'Epochs      : {cfg.STAGE1_EPOCHS}')
print(f'Batch size  : {cfg.STAGE1_BATCH_SIZE}')
print(f'LR          : {cfg.STAGE1_LR}')
print(f'Warmup steps: {cfg.STAGE1_WARMUP_STEPS}')
print(f'Max words   : {cfg.STAGE1_MAX_WORDS}')

In [ ]:
# â”€â”€ 6. Load dataset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# WikiText-103 + CoNLL-2003 train split (~1.2M sentences after filtering)
from src.data.dataset import PretrainDataset

train_dataset = PretrainDataset(split='train', max_words=cfg.STAGE1_MAX_WORDS, verbose=True)

In [ ]:
# â”€â”€ 7. Build models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import math
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# BERT teacher (frozen)
print('Loading BERT teacher ...')
bert_tokenizer = AutoTokenizer.from_pretrained(cfg.BERT_MODEL_NAME)
bert = AutoModel.from_pretrained(cfg.BERT_MODEL_NAME)
bert.eval()
for p in bert.parameters():
    p.requires_grad_(False)
bert = bert.to(DEVICE)
print(f'  BERT params: {sum(p.numel() for p in bert.parameters()):,}  (all frozen)')

# RetBERT student
retbert = RetBERT().to(DEVICE)
trainable = [p for p in retbert.parameters() if p.requires_grad]
print(f'  RetBERT trainable: {sum(p.numel() for p in trainable):,}')

In [ ]:
# â”€â”€ 8. Training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@torch.no_grad()
def bert_sentence_emb(batch_words):
    texts = [' '.join(w) for w in batch_words]
    enc = bert_tokenizer(
        texts, max_length=cfg.STAGE1_MAX_SUBWORDS,
        truncation=True, padding=True, return_tensors='pt'
    ).to(DEVICE)
    out = bert(**enc)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    return (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def make_scheduler(optimizer, warmup, total):
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        p = (step - warmup) / max(1, total - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))
    return LambdaLR(optimizer, lr_lambda)

loader = DataLoader(
    train_dataset,
    batch_size=cfg.STAGE1_BATCH_SIZE,
    shuffle=True,
    collate_fn=pretrain_collate,
    num_workers=2,
    pin_memory=True,
)

optimizer   = AdamW(trainable, lr=cfg.STAGE1_LR, weight_decay=0.01)
total_steps = cfg.STAGE1_EPOCHS * len(loader)
scheduler   = make_scheduler(optimizer, cfg.STAGE1_WARMUP_STEPS, total_steps)

loss_history = []
best_loss    = float('inf')

for epoch in range(1, cfg.STAGE1_EPOCHS + 1):
    retbert.train()
    epoch_loss = 0.0

    for step, batch_words in enumerate(loader, 1):
        z_bert    = bert_sentence_emb(batch_words)
        _, z_rb   = retbert(batch_words)
        loss      = stage1_loss(z_bert, z_rb)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        if step % 500 == 0:
            avg = epoch_loss / step
            lr  = scheduler.get_last_lr()[0]
            print(f'  E{epoch} {step}/{len(loader)}  loss={avg:.4f}  lr={lr:.2e}', flush=True)

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f'Epoch {epoch}/{cfg.STAGE1_EPOCHS}  avg_loss={avg:.4f}')

    if avg < best_loss:
        best_loss = avg
        torch.save(retbert.state_dict(), str(cfg.STAGE1_CKPT))
        print(f'  -> saved {cfg.STAGE1_CKPT}')

print(f'\nDone. Best loss: {best_loss:.4f}')

In [ ]:
# â”€â”€ 9. Loss curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.plot(range(1, len(loss_history)+1), loss_history, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Avg cosine distance loss')
plt.title('Stage 1 training loss')
plt.tight_layout()
plt.savefig('/kaggle/working/stage1_loss.png', dpi=100)
plt.show()

import os
ckpt = cfg.STAGE1_CKPT
print(f'Checkpoint: {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB)')
print('Download this file and upload as a Kaggle dataset for Stage 2.')